In [2]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/real_fake_news_dataset.zip")
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [14]:
!pip install langchain langchain-community ollama

  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached marshmallow-3.26.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 13.2 MB/s eta 0:00:00
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.0-py3-none-any.whl (7.8 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 15.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninsta

In [15]:
from langchain.prompts import PromptTemplate

template = """
You are a strict classifier. Your job is to read the news article below and classify it into ONLY ONE WORD: either "REAL" or "FAKE".

Do NOT provide any reasoning or explanation. Do NOT add any additional words, symbols, or punctuation.
Your response must be exactly one of these two words:
- REAL
- FAKE

Classification Criteria:
REAL News typically includes:
- Neutral or factual headlines
- Recognized news sources (e.g., *Reuters*, *BBC*, *The Washington Post*)
- Specific, verifiable information (data, quotes, evidence)
- Formal and objective language

FAKE News typically includes:
- Sensational or emotionally charged headlines
- Dubious or unnamed sources (e.g., *Before It’s News*, anonymous blogs)
- Vague or unverifiable claims
- Exaggerated or conspiratorial language

Examples:
1. REAL → Headline: "100,000 protest in the capital", Source: *The Washington Post*
2. FAKE → Headline: "Breaking: Government Hides the Truth About Aliens!", Source: *Before It's News*

Article:
Headline: {headline}
Text: {text}

Respond with only one word: REAL or FAKE
Your answer:
"""

prompt = PromptTemplate(
    input_variables=["headline", "text"],
    template=template
)

In [18]:
!pip install langchain_ollama

Using cached langchain_ollama-0.3.3-py3-none-any.whl (21 kB)


In [ ]:
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnableLambda

llm = OllamaLLM(
    model="granite3.3:8b",
    tempereture=0.001,
    top_k=2,
    top_p = 0.01
    )

chain = RunnableLambda(lambda inputs: llm.invoke(prompt.format(**inputs)))

# Testing model Granite
headline = df.loc[0, 'title']
text = df.loc[0, 'text']
response = chain.invoke({"headline": headline, "text": text})
print(response)